## README
You need 3 files to run this code:
- AnnGeno.ag file for genotypes and phenotypes. If you don't have this file run code xx
- burdens.zarr for gene burden scores. If you don't have this file run compute_burdens.ipynb
- PRS.parquet for PRS scores per phenotype. If the PRS columns have different column names, you will need a PRS_id to phenotype mapping file
- associations.parquet file with the all the gene-trait associations to test


In [ ]:
import sys
import yaml
import zarr
import pandas as pd
import numpy as np
from tqdm import tqdm
from anngeno import AnnGeno
import matplotlib.pyplot as plt
from plotnine import *


In [ ]:
# Get covariate corrected phenotypes
def cov_prs_correction(all_df, phenotypes, covariates, prs_pheno_map):
    # Initialize an empty DataFrame to store residuals
    # all_df.set_index('sample', inplace=True)
    cov_prs_corrected_phenos = pd.DataFrame(index=all_df.index) # Index is the sample ID

    # Perform linear regression for each phenotype
    for pheno in tqdm(phenotypes):
        # Drop NaN values for the current phenotype
        combined_df = all_df[[pheno] + covariates + [prs_pheno_map[pheno]]].dropna()
        y = combined_df[pheno]
        X = combined_df.drop(columns=[pheno])
        X = sm.add_constant(X)  # Add a constant term for the intercept

        # Fit the model
        model = sm.OLS(y, X).fit()

        # Save residuals
        residuals = pd.Series(model.resid, index=combined_df.index, name=pheno)
        cov_prs_corrected_phenos = pd.concat([cov_prs_corrected_phenos, residuals], axis=1)

    # Reset the index for the resulting DataFrame
    cov_prs_corrected_phenos.reset_index(inplace=True)
    # cov_prs_corrected_phenos.columns = ['sample'] + [f"{pheno}_cov_prs_corrected" for pheno in phenotypes]
    return cov_prs_corrected_phenos


In [ ]:
def pheno_burden_spearman(assoc_df, gt_df, annotation):
    rank_corr_list = []
    for trait in assoc_df.phenotype.unique():
        gene_list = list(assoc_df.query("phenotype == @trait").gene.astype(str))
        pheno = trait.replace(" ", "_")
        for gene in gene_list:
            try:
                correlation = gt_df[[gene, pheno]].dropna().corr(method='spearman').iloc[0, 1]
            except:
                print(f"Cannot compute correlation for {annotation}, {pheno}, {gene}. Error: {e}")
                correlation = np.nan
            
            rank_corr_list.append(
                pd.DataFrame({
                    'annotation': annotation,
                    'phenotype': trait,
                    'gene': gene,
                    'spearman_correlation': correlation
                }, index=[0])
            )
    return pd.concat(rank_corr_list)

In [ ]:
def compute_correlations(config_path, zarr_burdens_path, associations_df_path):

    with open(config_path) as f:
        config = yaml.safe_load(f)

    anngeno_file = config.get('anngeno_file')
    annotation_list = config.get('rare_variant_annotations')
    phenotypes = config.get('phenotypes_for_association_testing')
    covs = config.get('covariates')
    prs_pheno_map_file = config.get('prs_pheno_map_file')
    prs_file = config.get('prs_file')

    cov_pheno_df = pd.read_parquet(f"{anngeno_file}/phenotypes.parquet", columns=['sample'] + covs + phenotypes).set_index('sample')
    prs_df = pd.read_parquet(prs_file)
    prs_df = prs_df[prs_df.index.isin(pdf.index)]
    prs_pheno_map = pd.read_csv(prs_pheno_map_file)
    prs_pheno_map = dict(zip(prs_pheno_map["phenotype"], prs_pheno_map["pgs_id"]))
    all_df = pd.concat([cov_pheno_df, prs_df], axis=1)
    pheno_corrected_df = cov_prs_correction(all_df, phenotypes, covs, prs_pheno_map)

    zarr_group = zarr.open_group(zarr_burdens_path, mode="r")
    sample_list = zarr_group['samples'][:]
    gene_list = zarr_group["genes"][:]
    annotation_list = zarr_group["annotations"][:]

    assoc_df = pd.read_parquet(associations_df_path)
    rho_df_list = []
    for anno in tqdm(annotation_list):
        anno_idx = np.where(annotation_list == anno)[0][0]
        burdens = zarr_group["gene_burdens"][:, :, anno_idx]
        gt_df = pd.DataFrame(burdens, index=sample_list, columns=gene_list).merge(pheno_corrected_df, left_index=True, right_on='sample')
        rho_df_list.append(pheno_burden_spearman(assoc_df, gt_df, anno))

    rho_df = pd.concat(rho_df_list)
    return rho_df

In [ ]:
config_path = './deeprvat_config.yaml'
zarr_burdens_path = '/s/project/deeprvat/ukb_gym/debugging/gene_burdens.zarr'
associations_df_path = '/s/project/deeprvat/ukb_gym/161k_plof_associations.pq' 

rho_df = compute_correlations(config_path, zarr_burdens_path, associations_df_path)
rho_df

In [ ]:
rho_df.to_parquet('/s/project/deeprvat/ukb_gym/debugging/spearman_correlations.pq')

### Debugging

In [ ]:
rho_df.groupby('annotation')['spearman_correlation'].apply(lambda x: x.isna().sum()).sort_values()

In [ ]:
burden_file = "/s/project/deeprvat/ukb_gym/debugging/gene_burdens.zarr"
zarr_group = zarr.open_group(burden_file, mode="r")
sample_list = zarr_group['samples'][:]
gene_list = zarr_group["genes"][:]
annotation_list = zarr_group["annotations"][:]


In [ ]:
anno2test = 'Consequence_stop_lost'
anno_idx = np.where(annotation_list == anno2test)[0][0]
burdens = zarr_group["gene_burdens"][:, :, anno_idx]
g_df = pd.DataFrame(burdens, index=sample_list, columns=gene_list)
g_df

In [ ]:
g_df.sum().sort_values(ascending=False)

In [ ]:
g_df.sum().hist(bins=100)
plt.show()

## Make plots

In [ ]:
rank_corr_df = pd.read_parquet('/s/project/deeprvat/ukb_gym/debugging/spearman_correlations.pq')
rank_corr_df['abs_correlation'] = np.abs(rank_corr_df['spearman_correlation'])
rank_corr_df['median_abs_correlation'] = rank_corr_df.groupby('annotation')['abs_correlation'].transform('median')
rank_corr_df = rank_corr_df.sort_values('median_abs_correlation', ascending=False)
rank_corr_df['annotation'] = pd.Categorical(rank_corr_df['annotation'], categories=rank_corr_df['annotation'].unique(), ordered=True)
rank_corr_df

In [ ]:
config_path = './deeprvat_config.yaml'  # Or wherever your config file is

with open(config_path) as f:
    config = yaml.safe_load(f)

rare_variant_annotations_dict = config.get('rare_variant_annotations')

# Create a mapping from annotation name to category
annotation_category_map = {}
if rare_variant_annotations_dict:
    for category_name, annotations in rare_variant_annotations_dict.items():
        for annotation in annotations:
            annotation_category_map[annotation] = category_name

# Add a 'category' column to rank_corr_df based on the mapping
rank_corr_df['category'] = rank_corr_df['annotation'].map(annotation_category_map)
rank_corr_df

In [ ]:
(
    ggplot(rank_corr_df, aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    # scale_y_log10() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    facet_wrap('~category', scales='free_x') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(10, 10),
    )
)

In [ ]:
(
    ggplot(rank_corr_df, aes(x='annotation', y='abs_correlation')) +
    geom_boxplot() +
    theme_bw() +
    scale_y_sqrt() +
    ylab('|rank correlation|') +
    theme(
        axis_text_x=element_text(rotation=90, vjust=1),
        figure_size=(10, 10),
    )
)